Oracle AI Data Platform v1.0

Copyright © 2026, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# Landing zone, workflow 2: ingest

Copies new data from the landing zone into the managed Delta tables that workflow 1 created. Run it
as a workflow on a schedule, or after new files land.

**Which tables.** Only managed tables that carry the table property `landing_zone.source` are processed.
The property names the external table to read, so this notebook needs no list of tables: a folder
added later is picked up as soon as workflow 1 has created its tables.

**Only new files.** Every row records the file it came from in `_source_file`. Each run reads the
external table with `input_file_name()` and keeps only rows from files that are not in the managed
table yet, so a rerun inserts nothing twice.

**One table failing doesn't stop the others.** The run fails at the end if any table failed.

## 1 · Parameters

Set them here, or as job parameters on the workflow task (a job parameter wins).

In [ ]:
import re
from pyspark.sql import functions as F

def param(name, default):
    """Return the workflow job parameter `name` when it is set, otherwise `default`."""
    try:
        value = oidlUtils.parameters.getParameter(name, default)   # oidlUtils comes with the AIDP runtime
    except NameError:                                             # running interactively, not as a job
        value = default
    return default if value in (None, "") else value

def is_valid_name(name):
    """True for a plain catalog, schema or table name: letters, digits and _, not starting with a digit."""
    return re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name) is not None

def first_line(error):
    """The first useful line of an error message - Spark errors can run to pages."""
    lines = [line.strip(" :") for line in str(error).splitlines() if line.strip(" :")]
    # Errors raised in Java start with a generic "An error occurred while calling ..." line: skip it.
    useful = [line for line in lines if not line.startswith("An error occurred while calling")]
    return (useful or lines or [repr(error)])[0][:200]

MANAGED_CATALOG = param("MANAGED_CATALOG", "silver")   # where workflow 1 created the managed tables
SCHEMA          = param("SCHEMA", "")                  # one schema, or "" for every schema in the catalog

for name in (MANAGED_CATALOG, SCHEMA):
    if name and not is_valid_name(name):
        raise ValueError(f"{name!r} is not a valid catalog or schema name")

## 2 · Find the landing-zone tables

Every table in scope is checked for the `landing_zone.source` property; tables without it are ignored.
Workflow 1 always writes `<external catalog>.<same schema>.<same table>`, and only that shape is
accepted, so a property set by hand cannot point this workflow at some other table.

In [ ]:
def landing_zone_source(full_name):
    """The value of the landing_zone.source property, or None if the table does not have it."""
    properties = {row.key: row.value for row in spark.sql(f"SHOW TBLPROPERTIES {full_name}").collect()}
    return properties.get("landing_zone.source")

def is_expected_source(external_table, schema, table):
    """True when `external_table` is <some catalog>.<schema>.<table>, the shape workflow 1 writes.
    Spark resolves names case-insensitively, so the comparison ignores case too."""
    parts = external_table.split(".")
    return len(parts) == 3 and is_valid_name(parts[0]) and [p.lower() for p in parts[1:]] == [schema.lower(), table.lower()]

if SCHEMA:
    schemas = [SCHEMA]
    scope = f"{MANAGED_CATALOG}.{SCHEMA}"
else:
    schemas = [row[0] for row in spark.sql(f"SHOW SCHEMAS IN {MANAGED_CATALOG}").collect()]
    scope = f"{MANAGED_CATALOG} (all schemas)"

landing_tables = []     # (managed table, external table it is loaded from)
for schema in schemas:
    try:
        tables_in_schema = spark.sql(f"SHOW TABLES IN {MANAGED_CATALOG}.`{schema}`").collect()
    except Exception as error:     # a schema we cannot list must not stop the others
        print(f"   skipping schema {schema}: {first_line(error)}")
        continue
    for row in tables_in_schema:
        if row.isTemporary or not is_valid_name(schema) or not is_valid_name(row.tableName):
            continue               # workflow 1 only creates plain names
        managed_table = f"{MANAGED_CATALOG}.{schema}.{row.tableName}"
        try:
            external_table = landing_zone_source(managed_table)
        except Exception:          # a table we cannot inspect is not one of ours
            continue
        if not external_table:
            continue
        if is_expected_source(external_table, schema, row.tableName):
            landing_tables.append((managed_table, external_table))
        else:
            print(f"   ignoring {managed_table}: its landing_zone.source ({external_table}) is not "
                  f"<catalog>.{schema}.{row.tableName}")

print(f"{len(landing_tables)} landing-zone table(s) in {scope}")
for managed_table, external_table in landing_tables:
    print(f"   {external_table}  ->  {managed_table}")

## 3 · Ingest each table

For each table:

1. `REFRESH TABLE` on the external table, so files that landed since the last run are visible.
2. Insert the rows of files not loaded yet: `input_file_name()` tags each row with its file, and a
   left anti join against the files already in the managed table keeps only the new ones.
3. Compare row counts. Fewer managed rows than external rows is a **failure**. More managed rows is only
   reported: it means files were removed from the bucket after they were loaded.

In [ ]:
def target_column(field):
    """How to fill one column of the managed table from the new rows."""
    if field.name == "_source_file":
        return F.col("_source_file")
    if field.name == "_ingested_at":
        return F.current_timestamp().alias("_ingested_at")
    # Cast to the managed table's type: Delta does not widen types on insert
    # (an INT value into a BIGINT column fails with DELTA_FAILED_TO_MERGE_FIELDS).
    return F.col(field.name).cast(field.dataType)

def summary_row(table, status, note="", new_files=None, rows_inserted=None, managed_rows=None,
                external_rows=None):
    return {"table": table, "status": status, "new_files": new_files, "rows_inserted": rows_inserted,
            "managed_rows": managed_rows, "external_rows": external_rows, "note": note}

def ingest(managed_table, external_table):
    """Insert the rows of new files from `external_table` into `managed_table`. Returns a summary row."""
    spark.sql(f"REFRESH TABLE {external_table}")

    # Every external row, tagged with the file it came from.
    tagged_rows = spark.table(external_table).withColumn("_source_file", F.input_file_name())
    # The files the managed table already holds.
    loaded_files = spark.table(managed_table).select("_source_file").distinct()
    # Keep only rows whose file is not loaded yet, shaped like the managed table, column by column.
    new_rows = (tagged_rows.join(loaded_files, on="_source_file", how="left_anti")
                           .select(*[target_column(field) for field in spark.table(managed_table).schema]))

    new_files = new_rows.select("_source_file").distinct().count()
    rows_before = spark.table(managed_table).count()
    if new_files:
        new_rows.write.insertInto(managed_table)    # by position: the columns are in the table's order

    managed_rows = spark.table(managed_table).count()
    external_rows = spark.table(external_table).count()
    rows_inserted = managed_rows - rows_before
    if managed_rows < external_rows:
        status, note = "FAILED", f"{external_rows - managed_rows} external row(s) missing from the managed table"
    elif managed_rows > external_rows:
        status, note = "ok", "more managed rows than external rows: files were removed from the bucket"
    else:
        status, note = "ok", ""
    return summary_row(managed_table, status, note, new_files, rows_inserted, managed_rows, external_rows)

In [ ]:
results = []
for managed_table, external_table in landing_tables:
    print(f"ingesting {managed_table} ...", flush=True)
    try:
        results.append(ingest(managed_table, external_table))
    except Exception as error:      # one table failing must not stop the others
        results.append(summary_row(managed_table, "FAILED", first_line(error)))

## 4 · Summary

In [ ]:
if results:
    spark.createDataFrame(results, "table STRING, status STRING, new_files LONG, rows_inserted LONG, "
                                   "managed_rows LONG, external_rows LONG, note STRING").show(100, truncate=False)
else:
    print("Nothing to ingest: no table carries landing_zone.source. Run workflow 1 (setup) first.")

failed_tables = [row["table"] for row in results if row["status"] == "FAILED"]
if failed_tables:
    raise RuntimeError(f"{len(failed_tables)} table(s) failed: {failed_tables}")